# Fallbacks and Model Failover

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/08-fallbacks-model-failover.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic openai

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your primary AI model is unavailable. Maybe the Anthropic API is having an incident. Maybe your account hit its rate limit and the next request slot is 20 minutes away. Maybe the model is responding but taking 45 seconds per request, which is functionally unavailable for a user-facing product.

The naive response is to return an HTTP 500 and hope users try again. In a user-facing product, that is catastrophic: a competitor with a working service gets every user who hit your 500. In an internal tool, it means engineers stop trusting the AI features and route around them. Either way, you have trained your users that the AI is unreliable.

The production response is a fallback chain: primary mo...

## The Concept

### The Fallback Chain

```
User Request
     |
     v
[1] Primary: Claude (claude-3-5-haiku-20241022)
     |  timeout=10s; errors: network, 5xx, 429
     v (on failure or timeout)
[2] Secondary: OpenAI (gpt-4o-mini)
     |  timeout=15s; errors: network, 5xx, 429
     v (on failure or timeout)
[3] Cache: Return a cached response if one exists for this query (or similar)
     |  cache hit: return immediately; cache miss: continue
     v (on cache miss)
[4] Degradation: Return a static message explaining the outage
     |  always succeeds; always returns something
     v
Response to User
```

Ea...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Fallbacks and Model Failover for AI services.

Implements a FallbackChain with four tiers:
  1. Primary model (Claude, short timeout)
  2. Secondary model (OpenAI, longer timeout)
  3. Cache (in-memory, TTL-based)
  4. Degradation message (static; always returns something)

Usage:
    ANTHROPIC_API_KEY=sk-ant-... OPENAI_API_KEY=sk-... python main.py

    # With broken primary (timeout=0.001 forces fallback):
    ANTHROPIC_API_KEY=invalid OPENAI_API_KEY=sk-... python main.py
"""

import hashlib
import os
import signal
import sys
import time
from dataclasses import dataclass, field
from typing import Any

### Timeout utility (POSIX only; works on Linux and macOS)

In [ ]:
class CallTimeoutError(Exception):
    """Raised when a model call exceeds its per-model timeout."""
    pass


def _alarm_handler(signum, frame):
    raise CallTimeoutError("Model call timed out")


def call_with_timeout(fn, timeout: float) -> Any:
    """
    Execute fn and raise CallTimeoutError if it takes longer than timeout seconds.

    Uses POSIX SIGALRM. For Windows compatibility, replace with
    concurrent.futures.ThreadPoolExecutor with a future.result(timeout=timeout).
    """
    signal.signal(signal.SIGALRM, _alarm_handler)
    signal.alarm(max(1, int(timeout)))  # alarm takes integer seconds
    try:
        return fn()
    finally:
        signal.alarm(0)  # cancel the alarm

### Model config

In [ ]:
@dataclass
class ModelConfig:
    """Configuration for a single model tier in the fallback chain."""
    provider: str    # "anthropic" or "openai"
    model: str       # provider-specific model ID
    timeout: float   # seconds; raise CallTimeoutError if exceeded
    max_tokens: int = 1024

### FallbackChain

In [ ]:
class FallbackChain:
    """
    Tries AI models in priority order. Falls back to a response cache,
    then to a static degradation message.

    The chain always returns a response -- it never raises an exception to
    the caller. The 'tier' field in the result indicates which tier answered.

    Stats are tracked per tier for operational monitoring:
        chain.stats -> {"primary": N, "fallback": N, "cache": N, "degraded": N}
    """

    def __init__(
        self,
        models: list[ModelConfig],
        degradation_message: str = (
            "The AI assistant is temporarily unavailable. "
            "Please try again in a few minutes."
        ),
        cache_ttl: float = 300.0,
    ):
        if not models:
            raise ValueError("FallbackChain requires at least one model config")
        self.models = models
        self.degradation_message = degradation_message
        self.cache_ttl = cache_ttl
        self._cache: dict[str, dict] = {}
        self._stats: dict[str, int] = {
            "primary": 0,
            "fallback": 0,
            "cache": 0,
            "degraded": 0,
        }

    @property
    def stats(self) -> dict[str, int]:
        return dict(self._stats)

    def generate(self, prompt: str) -> dict:
        """
        Try each model in order. Return the first successful response.

        Returns a dict with:
          text          - the response text
          tier          - which tier answered: primary / fallback / cache / degraded
          model         - model ID that answered (or "cache" / "none")
          latency_seconds - wall-clock time for the model call (0 for cache/degraded)
        """
        for i, config in enumerate(self.models):
            tier_name = "primary" if i == 0 else "fallback"
            try:
                start = time.monotonic()
                text = call_with_timeout(
                    lambda c=config: self._call_model(c, prompt),
                    timeout=config.timeout,
                )
                elapsed = time.monotonic() - start
                self._put_in_cache(prompt, text)
                self._stats[tier_name] += 1
                return {
                    "text": text,
                    "tier": tier_name,
                    "model": config.model,
                    "latency_seconds": round(elapsed, 3),
                }

            except Exception as e:
                print(
                    f"[FallbackChain] Tier {i+1} ({config.model}) failed: "
                    f"{type(e).__name__}: {e}",
                    file=sys.stderr,
                )
                continue

        # All models failed: try cache
        cached_text = self._get_from_cache(prompt)
        if cached_text is not None:
            self._stats["cache"] += 1
            return {
                "text": cached_text,
                "tier": "cache",
                "model": "cache",
                "latency_seconds": 0.0,
            }

        # Nothing worked: return degradation message
        self._stats["degraded"] += 1
        return {
            "text": self.degradation_message,
            "tier": "degraded",
            "model": "none",
            "latency_seconds": 0.0,
        }

    # ---------------------------------------------------------------------------
    # Cache helpers
    # ---------------------------------------------------------------------------

    def _cache_key(self, prompt: str) -> str:
        return hashlib.sha256(prompt.encode()).hexdigest()[:16]

    def _get_from_cache(self, prompt: str) -> str | None:
        entry = self._cache.get(self._cache_key(prompt))
        if entry and (time.monotonic() - entry["ts"]) < self.cache_ttl:
            return entry["text"]
        return None

    def _put_in_cache(self, prompt: str, text: str) -> None:
        self._cache[self._cache_key(prompt)] = {
            "text": text,
            "ts": time.monotonic(),
        }

    # ---------------------------------------------------------------------------
    # Model callers
    # ---------------------------------------------------------------------------

    def _call_model(self, config: ModelConfig, prompt: str) -> str:
        if config.provider == "anthropic":
            return self._call_anthropic(config, prompt)
        elif config.provider == "openai":
            return self._call_openai(config, prompt)
        else:
            raise ValueError(f"Unknown provider: {config.provider!r}")

    def _call_anthropic(self, config: ModelConfig, prompt: str) -> str:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("ANTHROPIC_API_KEY not set")
        client = anthropic.Anthropic(api_key=api_key)
        msg = client.messages.create(
            model=config.model,
            max_tokens=config.max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return msg.content[0].text

    def _call_openai(self, config: ModelConfig, prompt: str) -> str:
        from openai import OpenAI
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise RuntimeError("OPENAI_API_KEY not set")
        client = OpenAI(api_key=api_key)
        resp = client.chat.completions.create(
            model=config.model,
            max_tokens=config.max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.choices[0].message.content

### Demo

In [ ]:
def main():
    chain = FallbackChain(
        models=[
            ModelConfig(
                provider="anthropic",
                model="claude-3-5-haiku-20241022",
                timeout=10.0,
            ),
            ModelConfig(
                provider="openai",
                model="gpt-4o-mini",
                timeout=15.0,
            ),
        ],
        degradation_message=(
            "The AI assistant is temporarily unavailable. "
            "Please try again in a few minutes."
        ),
        cache_ttl=300.0,
    )

    prompts = [
        "Explain fallback chains in one sentence.",
        "Explain fallback chains in one sentence.",  # second call hits cache
        "What is the difference between a timeout and a 503 error?",
    ]

    for prompt in prompts:
        print(f"\nPrompt: {prompt[:60]}...")
        result = chain.generate(prompt)
        print(f"  Tier:    {result['tier']}")
        print(f"  Model:   {result['model']}")
        print(f"  Latency: {result['latency_seconds']:.3f}s")
        print(f"  Text:    {result['text'][:100]}...")

    print(f"\nStats: {chain.stats}")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. Which fallback trigger applies here, and what tier should handle this request instead?**

- A. Error-based failover: Claude returned a 503 error, so the secondary model should try.
- B. Timeout-based failover: Claude exceeded the 10-second timeout, so the secondary model should try. The failure is slowness, not an error code.
- C. No fallback is needed. The timeout means the request was cancelled; the user should retry manually.
- D. Cache-based fallover: the slow response should be replaced by a cached response from a previous request.

**2. What does the fallback chain return, and is this the correct behavior?**

- A. A 503 error to the caller, because both model tiers failed and the service should be transparent about unavailability.
- B. The cached response from 3 minutes ago, with tier='cache'. This is correct: the user gets a response that was accurate 3 minutes ago, which is better than an error.
- C. A new request is sent to both models simultaneously to reduce the chance of both failing.
- D. The degradation message, because cached responses may be stale and should not be served.

**3. What is the user-facing impact of a 60-second primary timeout compared to a 10-second timeout?**

- A. No impact. Timeouts only affect infrastructure costs, not user experience.
- B. With a 60-second timeout, users wait up to 60 seconds before the fallback chain tries the secondary model. On a slow primary, users experience 60-second hangs before getting any response. A 10-second timeout means users wait at most 10 seconds before the faster secondary may answer.
- C. A longer timeout improves response quality because the model has more time to think.
- D. A 60-second timeout reduces API costs because fewer fallback calls are made to the secondary provider.

**4. What does a 5% fallback rate actually indicate, and why is it worth investigating even though degraded rate is low?**

- A. Nothing unusual. 5% fallback is within normal operating range for AI services.
- B. The primary model is failing or timing out 5% of the time. This means 1 in 20 requests is being served by the secondary model instead of the intended primary. Users may notice quality differences, and the team is paying for secondary provider capacity that should be exceptional, not routine.
- C. The 5% fallback rate means the cache is working correctly and saving 5% of requests from hitting the primary.
- D. A 5% fallback rate is expected and intentional -- it means the load balancer is distributing traffic between providers.

**5. What is the problem with falling back to GPT-4o-mini in this scenario?**

- A. No problem. GPT-4o-mini may have a larger context limit and the request may succeed.
- B. A 400 error means the request itself is malformed (the system prompt is too long). The same request will also fail on GPT-4o-mini with a similar error. Falling back wastes time and API quota without fixing the root problem.
- C. The fallback should work because GPT-4o-mini uses a different tokenizer and the prompt may be within its limit.
- D. The fallback chain should retry the primary model three times before trying the secondary.

**6. Why is a 24-hour cache TTL inappropriate for this use case, and what TTL would be more appropriate?**

- A. A 24-hour TTL is fine. Stock prices are public information and accuracy is not critical for most users.
- B. Stock prices change every second during market hours. A 24-hour cached response to 'What is AAPL's current price?' will be wildly inaccurate. For real-time financial data, the cache TTL should be 0 (disabled) or at most a few seconds.
- C. The problem with 24 hours is that it is longer than the model's training data cutoff, causing hallucinations.
- D. Cache TTL only affects latency, not accuracy. A 24-hour TTL does not cause incorrect responses.

**7. What is the argument for keeping the degradation message as a 200 response versus returning a 500 error?**

- A. Returning 500 is always better because it is semantically correct -- the service failed.
- B. A 200 response with a degradation message keeps the service 'up' from the perspective of health checks and SLO calculations. More importantly, it allows the client to display a human-readable message without custom error handling. A 500 requires the frontend to have error UI; a 200 with tier='degraded' allows graceful in-line messaging.
- C. The degradation message should be a 503 Service Unavailable, not a 200 or 500.
- D. Both approaches are equivalent. The status code does not affect user experience.

_Answers are in `checks.json` in the lesson directory._